In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# keras.layers.Dense
# keras.models.Sequential
# tf.keras.optimizers.RMSprop(0.001)


In [ ]:
tf.keras.utils.set_random_seed(42)

dataset = pd.read_csv('sonar.csv', header=None)
dataset.tail()

train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset[60])
train_dataset, validation_dataset = train_test_split(train_dataset, test_size=0.2, random_state=42, stratify=train_dataset[60])

train_labels = train_dataset.pop(60).map({'R': 0, 'M': 1})
validation_labels = validation_dataset.pop(60).map({'R': 0, 'M': 1})
test_labels = test_dataset.pop(60).map({'R': 0, 'M': 1})

train_labels_cat = keras.utils.to_categorical(train_labels)
validation_labels_cat = keras.utils.to_categorical(validation_labels)
test_labels_cat = keras.utils.to_categorical(test_labels)

train_dataset = train_dataset.astype('float32')
validation_dataset = validation_dataset.astype('float32')
test_dataset = test_dataset.astype('float32')

learning_rates = [0.1, 0.01, 0.001, 0.0001]

def build_model(learning_rate):
    model = keras.Sequential([
        layers.Dense(32, activation='relu', input_shape=[len(train_dataset.keys())]),
        layers.Dense(16, activation='relu'),
        layers.Dense(2, activation='softmax')
    ])

    optimizer = tf.keras.optimizers.RMSprop(learning_rate)

    model.compile(optimizer=optimizer,
                  loss=tf.keras.losses.CategoricalCrossentropy(),
                  metrics=['accuracy'])
    return model

validation_results = []
for learning_rate in learning_rates:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    model = build_model(learning_rate)
    history = model.fit(
        train_dataset,
        train_labels_cat,
        epochs=100,
        batch_size=16,
        validation_data=(validation_dataset, validation_labels_cat),
        verbose=0,
    )
    best_validation_accuracy = max(history.history['val_accuracy'])
    best_epoch = history.history['val_accuracy'].index(best_validation_accuracy) + 1
    validation_results.append({
        'learning_rate': learning_rate,
        'best_validation_accuracy': best_validation_accuracy,
        'best_epoch': best_epoch,
    })

results_df = pd.DataFrame(validation_results)
results_df

best_result = sorted(
    validation_results,
    key=lambda result: (-result['best_validation_accuracy'], result['learning_rate']),
)[0]
best_learning_rate = best_result['learning_rate']
print(f'Najbolji learning_rate: {best_learning_rate}')
print(f"Najbolja validaciona tacnost: {best_result['best_validation_accuracy']:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(results_df['learning_rate'].astype(str), results_df['best_validation_accuracy'], marker='o')
plt.xlabel('RMSprop learning_rate')
plt.ylabel('Najbolja validaciona tacnost')
plt.grid(True)
plt.show()

train_dataset_full = pd.concat([train_dataset, validation_dataset])
train_labels_full_cat = np.concatenate([train_labels_cat, validation_labels_cat])

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
final_model = build_model(best_learning_rate)
final_history = final_model.fit(
    train_dataset_full,
    train_labels_full_cat,
    epochs=100,
    batch_size=16,
    verbose=0,
)

test_loss, test_accuracy = final_model.evaluate(test_dataset, test_labels_cat, verbose=0)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
